In [2]:
import os
import sys
from os import path
sys.path.insert(0, "/home/thomasb")
import json
import h5py
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone
#import sat_utils as su
import importlib
#from albatros_analysis.src.utils import orbcomm_utils as outils
from albatros_analysis.src.utils import baseband_utils as butils
#import helper_discrepancies as hd
#importlib.reload(su)
importlib.reload(butils)
#importlib.reload(hd)

<module 'albatros_analysis.src.utils.baseband_utils' from '/home/thomasb/albatros_analysis/src/utils/baseband_utils.py'>

In [19]:
# a batch is just a different name for a period of continuous data. they usually last about a day
# they are index by the starting unix time, so set it here to access the data
batch_start_ts = 1753132820

In [20]:
# define some paths
path_taus = f'/scratch/thomasb/batch_{batch_start_ts}/fine_timing/timing_solution_15may.h5'
path_taus_testing = f'/scratch/thomasb/batch_{batch_start_ts}_testing/fine_timing/timing_solution_15may.h5'
path_UTC = f'/scratch/thomasb/batch_{batch_start_ts}/timing_discrepancies/times_all.json'

# spectra are our reliable proxy for time
# we have a fit for spectra to UTC time stored somewhere. extract it here
with open(path_UTC, 'r') as f:
    data_UTC = json.load(f)
    UTC_per_spec = data_UTC['fit']["UTC_per_spec"]
    UTC_offset = data_UTC['fit']["UTC_offset"]
print('UTC seconds per spectrum', UTC_per_spec)
print('UTC offset', UTC_offset)

#makes reading file easier
map_blines = {0:'MARS 1-2', 1:'MARS 1-4', 2:'MARS 1-5', 3:'MARS 1-6', 4:'MARS 1-7', 5:'MARS 1-8'}

UTC seconds per spectrum 1.6383993640972462e-05
UTC offset 1753113682.1941924


In [21]:
# some parameters 
# these depend on which batch we're looking at, and are only guaranteed to be correct for the batch given as an example here
# stuff will be made cleaner soon.
nvis = 120
nant = 7
nbl = int((nant-1)*nant/2)
acclen = 1024
osamp = 64
int_spec = osamp*acclen
T_SPECTRA = 4096/250e6
verbose = False
print('nbl:', nbl)
print('BB spectra per vis:', int_spec)

nbl: 21
BB spectra per vis: 65536


# Basic Formatting

For the time being, all data is in .h5 files, indexed by the long and annoying file name that corresponds to the pass data. We can make everybody's life easier by unpacking it cleanly.

In [19]:
data_dictionary1 = {}
data_dictionary1['baseline map'] = map_blines

In [20]:
pass_ctr = 0
with h5py.File(path_taus, 'r') as f:
    for name, obj in f.items():
        #make a sub-dictionary
        data_dictionary1[f'p{pass_ctr}'] = {}
        #specnum stuff
        start_spec = obj['taus'].attrs['starting_specnum']
        #beware: we want the CENTRAL spectrum number, not the STARTING one, we accumulate through entire integration time
        spectra = np.arange(int(start_spec+int_spec/2), int(start_spec + (nvis+1/2)*int_spec), int_spec)

        #taus stuff
        taus_old = obj['taus'][:]

        data_dictionary1[f'p{pass_ctr}']['spectra'] = spectra
        data_dictionary1[f'p{pass_ctr}']['taus'] = taus_old
        pass_ctr += 1

In [23]:
print(data_dictionary1.keys())

dict_keys(['baseline map', 'p0', 'p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p8', 'p9', 'p10', 'p11', 'p12'])


In [24]:
print(data_dictionary1['p1']['taus'].shape)

(6, 120)


# Explicit Baseline Taus

If you want all baselines to have the taus expressed

In [ ]:
data_dictionary2 = {}
data_dictionary2['baseline map'] = map_blines

In [ ]:
pass_ctr = 0
with h5py.File(path_taus, 'r') as f:
    for name, obj in f.items():
        #make a sub-dictionary
        data_dictionary2[f'p{pass_ctr}'] = {}
        #specnum stuff
        start_spec = obj['taus'].attrs['starting_specnum']
        #beware: we want the CENTRAL spectrum number, not the STARTING one, we accumulate through entire integration time
        spectra = np.arange(int(start_spec+int_spec/2), int(start_spec + (nvis+1/2)*int_spec), int_spec)

        #taus stuff
        taus_old = obj['taus'][:]
        taus_new = np.zeros((nbl, nvis))
        bl_ctr = 0
        for i in range(nant):
            if i == 0:
                taus_i = np.zeros(nvis)
            else:
                taus_i = taus_old[i-1,:]
            for j in range(i+1, nant):
                taus_j = taus_old[j-1,:]
                #convention is ref-nref, so delays are already in that form. 
                #sanity check: for ref ant we want the delays to stay same
                taus_new[bl_ctr, :] = taus_j - taus_i

                if verbose:
                    print(f'\nBaseline {bl_ctr}')
                    print('old i', taus_i[0])
                    print('old j', taus_j[0])
                    print('new:', taus_new[bl_ctr,0])
                bl_ctr+=1

        data_dictionary2[f'p{pass_ctr}']['spectra'] = spectra
        data_dictionary2[f'p{pass_ctr}']['taus'] = taus_new
        pass_ctr += 1

In [17]:
print(data_dictionary['p1']['taus'].shape)

(21, 120)


# New Format for New Function
The idea here is to store taus in a much more efficient way such that it's easier to just open up and look at.
In principle, get_batch_fineiming.py should save two copies of the timing solution: one as a debug that separates out pulses, and one as straight up data that concatenates them all together.

In [22]:
pass_ctr = 0
with h5py.File(path_taus, 'r') as f:
    for name, obj in f.items():
        if pass_ctr == 0:
            data = obj['taus'][:]
            n_nonref_ant, nvis = data.shape
        pass_ctr +=1

npasses = pass_ctr
print(npasses)
print(n_nonref_ant)
print(nvis)
ntimes = npasses*nvis

10
6
120


In [23]:
spectra = np.zeros(ntimes)
taus = np.zeros((n_nonref_ant, ntimes))

In [24]:
pass_ctr = 0
with h5py.File(path_taus, 'r') as f:
    for name, obj in f.items():
        #specnum stuff
        start_spec = obj['taus'].attrs['starting_specnum']
        #beware: we want the CENTRAL spectrum number, not the STARTING one, we accumulate through entire integration time
        s = np.arange(int(start_spec+int_spec/2), int(start_spec + (nvis+1/2)*int_spec), int_spec)
        spectra[pass_ctr*nvis: (pass_ctr+1)*nvis] = s
        #taus stuff
        taus_old = obj['taus'][:]
        print('taus shape', taus_old.shape)
        taus[:, pass_ctr*nvis: (pass_ctr+1)*nvis] = taus_old
        pass_ctr += 1

taus shape (6, 120)
taus shape (6, 120)
taus shape (6, 120)
taus shape (6, 120)
taus shape (6, 120)
taus shape (6, 120)
taus shape (6, 120)
taus shape (6, 120)
taus shape (6, 120)
taus shape (6, 120)


In [25]:
with h5py.File(f'/scratch/thomasb/timing_solution/batch_{batch_start_ts}.h5', "w") as f:
    f.create_dataset("spectra", data=spectra)
    f.create_dataset("taus", data=taus)